# Additional experiments to fully satisfy all reviewer comments:
1. TCN comparison (R1-5) - actual architecture comparison
2. GRU comparison - another recurrent baseline
3. Transformer comparison - scaled dot-product attention
4. SHAP sample validation (R1-1) - compare 20 vs 40 sample CIs

In [ ]:
import numpy as np
import pandas as pd
import pickle
import time
import json
import warnings
import os
import sys
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import (LSTM, GRU, Dense, Dropout, Layer,
                                      Conv1D, GlobalAveragePooling1D,
                                      LayerNormalization, MultiHeadAttention,
                                      Flatten, Add)
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping

results_dir = Path('../results')
tables_dir = results_dir / 'tables'

# Load data
print("Loading data...")
X_train_full = np.load(results_dir / 'X_train.npy')
X_test = np.load(results_dir / 'X_test.npy')
y_train_full = np.load(results_dir / 'y_train.npy')
y_test = np.load(results_dir / 'y_test.npy')

val_size = int(0.2 * len(X_train_full))
X_val = X_train_full[-val_size:]
y_val = y_train_full[-val_size:]
X_train = X_train_full[:-val_size]
y_train = y_train_full[:-val_size]

print(f"Data loaded. X_train: {X_train.shape}, X_test: {X_test.shape}")
sys.stdout.flush()

## EXPERIMENT 1: TCN MODEL (R1-5)

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 1: TEMPORAL CONVOLUTIONAL NETWORK (TCN)")
print("="*70)
sys.stdout.flush()

def build_tcn_model(input_shape, num_filters=32, kernel_size=3, dilations=[1, 2, 4], dropout_rate=0.2):
    """TCN with dilated causal convolutions."""
    inputs = Input(shape=input_shape)
    x = inputs
    for d in dilations:
        conv = Conv1D(filters=num_filters, kernel_size=kernel_size,
                      padding='causal', dilation_rate=d, activation='relu')(x)
        conv = Dropout(dropout_rate)(conv)
        # Residual connection if shapes match
        if x.shape[-1] == num_filters:
            x = Add()([x, conv])
        else:
            x = conv
    x = GlobalAveragePooling1D()(x)
    outputs = Dense(1)(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

## EXPERIMENT 2: GRU MODEL

In [ ]:
def build_gru_model(input_shape):
    """GRU baseline - another recurrent architecture."""
    model = Sequential([
        GRU(20, input_shape=input_shape),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

## EXPERIMENT 3: TRANSFORMER MODEL (R1-5)

In [ ]:
def build_transformer_model(input_shape, num_heads=2, ff_dim=32, dropout_rate=0.2):
    """Simple Transformer encoder for time series."""
    inputs = Input(shape=input_shape)
    # Multi-head self-attention
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=input_shape[-1])(inputs, inputs)
    attn_output = Dropout(dropout_rate)(attn_output)
    x = LayerNormalization()(inputs + attn_output)
    # Feed-forward
    ff = Dense(ff_dim, activation='relu')(x)
    ff = Dense(input_shape[-1])(ff)
    ff = Dropout(dropout_rate)(ff)
    x = LayerNormalization()(x + ff)
    x = GlobalAveragePooling1D()(x)
    outputs = Dense(1)(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

## EXPERIMENT 4: ATTENTION-LSTM (re-train for fair comparison)

In [ ]:
class AttentionLayer(Layer):
    def __init__(self, units=20, return_sequences=True, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.return_sequences = return_sequences
    def build(self, input_shape):
        self.W = self.add_weight(name='attention_weight', shape=(input_shape[-1], self.units), initializer='glorot_uniform')
        self.b = self.add_weight(name='attention_bias', shape=(self.units,), initializer='zeros')
        self.u = self.add_weight(name='attention_vector', shape=(self.units,), initializer='glorot_uniform')
        super().build(input_shape)
    def call(self, inputs):
        score = tf.nn.tanh(tf.tensordot(inputs, self.W, axes=1) + self.b)
        aw = tf.nn.softmax(tf.tensordot(score, self.u, axes=1), axis=1)
        aw_exp = tf.expand_dims(aw, -1)
        ctx = tf.reduce_sum(inputs * aw_exp, axis=1)
        if self.return_sequences:
            return ctx, tf.squeeze(aw_exp, -1)
        return ctx
    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units, 'return_sequences': self.return_sequences})
        return config

def build_attention_lstm(input_shape):
    inputs = Input(shape=input_shape)
    lstm_out = LSTM(20, return_sequences=True)(inputs)
    ctx, attn_w = AttentionLayer(units=20)(lstm_out)
    outputs = Dense(1)(ctx)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

## RUN ALL ARCHITECTURE COMPARISONS

In [ ]:
SEEDS = [42, 123, 456, 789]
input_shape = (X_train.shape[1], X_train.shape[2])
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Use a reasonable training subset for fair comparison
train_sub = min(200000, len(X_train))
np.random.seed(42)
train_idx = np.random.choice(len(X_train), train_sub, replace=False)
X_tr = X_train[train_idx]
y_tr = y_train[train_idx]

architectures = {
    'Vanilla LSTM': lambda: Sequential([
        LSTM(20, input_shape=input_shape), Dropout(0.2), Dense(1)
    ]),
    'GRU': lambda: build_gru_model(input_shape),
    'Attention-LSTM (Bahdanau)': lambda: build_attention_lstm(input_shape),
    'TCN (dilated causal)': lambda: build_tcn_model(input_shape),
    'Transformer (2-head)': lambda: build_transformer_model(input_shape),
}

all_arch_results = []

for arch_name, build_fn in architectures.items():
    print(f"\n--- {arch_name} ---")
    sys.stdout.flush()
    metrics = {'rmse': [], 'mae': [], 'r2': []}
    converged = 0
    train_times = []

    for seed in SEEDS:
        np.random.seed(seed)
        tf.random.set_seed(seed)

        model = build_fn()
        if arch_name == 'Vanilla LSTM':
            model.compile(optimizer='adam', loss='mse', metrics=['mae'])

        start = time.perf_counter()
        model.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                  epochs=50, batch_size=256,
                  callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
                  verbose=0)
        train_time = time.perf_counter() - start
        train_times.append(train_time)

        y_pred = model.predict(X_test, verbose=0).flatten()
        rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
        mae = float(mean_absolute_error(y_test, y_pred))
        r2 = float(r2_score(y_test, y_pred))

        metrics['rmse'].append(rmse)
        metrics['mae'].append(mae)
        metrics['r2'].append(r2)
        if r2 > 0:
            converged += 1

        print(f"  Seed {seed}: MAE={mae:.5f}, R²={r2:.4f}, Time={train_time:.1f}s")
        sys.stdout.flush()

    result = {
        'Architecture': arch_name,
        'Conv_Rate': f"{converged}/{len(SEEDS)}",
        'Mean_MAE': float(np.mean(metrics['mae'])),
        'Std_MAE': float(np.std(metrics['mae'])),
        'Mean_RMSE': float(np.mean(metrics['rmse'])),
        'Std_RMSE': float(np.std(metrics['rmse'])),
        'Mean_R2': float(np.mean(metrics['r2'])),
        'Std_R2': float(np.std(metrics['r2'])),
        'Best_R2': float(max(metrics['r2'])),
        'Worst_R2': float(min(metrics['r2'])),
        'Mean_Train_Time_s': float(np.mean(train_times)),
        'Params': model.count_params()
    }
    all_arch_results.append(result)
    print(f"  Summary: Conv={result['Conv_Rate']}, MAE={result['Mean_MAE']:.5f}±{result['Std_MAE']:.5f}, "
          f"R²={result['Mean_R2']:.4f}±{result['Std_R2']:.4f}, Params={result['Params']}")
    sys.stdout.flush()

pd.DataFrame(all_arch_results).to_csv(tables_dir / 'architecture_comparison.csv', index=False)
print("\n✅ Architecture comparison saved")
sys.stdout.flush()

## EXPERIMENT 5: SHAP SAMPLE SIZE VALIDATION (R1-1)

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 5: SHAP SAMPLE SIZE VALIDATION")
print("="*70)
sys.stdout.flush()

# Load existing SHAP values to validate bootstrap stability
try:
    shap_values = np.load(results_dir / 'shap_values_kernel.npy')
    print(f"Loaded SHAP values: {shap_values.shape}")

    # Original 20-sample bootstrap
    n_bootstrap = 1000
    n_samples = shap_values.shape[0]  # should be 20

    # Compute feature importance from full 20 samples
    feature_importance_20 = np.mean(np.abs(shap_values), axis=(0, 2))  # avg over samples and timesteps

    # Bootstrap with subsets: compare 10 vs 15 vs 20 samples
    subset_sizes = [10, 15, 20] if n_samples >= 20 else [n_samples]
    shap_validation = []

    for ss in subset_sizes:
        boot_means = []
        for b in range(n_bootstrap):
            idx = np.random.choice(ss, ss, replace=True)
            boot_sample = shap_values[idx]
            boot_means.append(np.mean(np.abs(boot_sample), axis=(0, 2)))

        boot_means = np.array(boot_means)
        ci_lower = np.percentile(boot_means, 2.5, axis=0)
        ci_upper = np.percentile(boot_means, 97.5, axis=0)
        ci_width = ci_upper - ci_lower
        mean_vals = np.mean(boot_means, axis=0)
        relative_ci = ci_width / (mean_vals + 1e-10) * 100

        # Check if feature rankings are stable
        ranking_stability = []
        for b in range(n_bootstrap):
            rank = np.argsort(boot_means[b])[::-1]
            ranking_stability.append(rank)
        ranking_stability = np.array(ranking_stability)
        top1_stability = np.mean(ranking_stability[:, 0] == ranking_stability[0, 0]) * 100
        top3_stability = np.mean([np.array_equal(r[:3], ranking_stability[0, :3]) for r in ranking_stability]) * 100

        result = {
            'Sample_Size': ss,
            'Mean_CI_Width': float(np.mean(ci_width)),
            'Max_Relative_CI_Pct': float(np.max(relative_ci)),
            'Mean_Relative_CI_Pct': float(np.mean(relative_ci)),
            'Top1_Ranking_Stability_Pct': float(top1_stability),
            'Top3_Ranking_Stability_Pct': float(top3_stability),
        }
        shap_validation.append(result)
        print(f"  n={ss}: Mean CI width={result['Mean_CI_Width']:.6f}, "
              f"Rel CI={result['Mean_Relative_CI_Pct']:.1f}%, "
              f"Top-1 stable={top1_stability:.0f}%, Top-3 stable={top3_stability:.0f}%")
        sys.stdout.flush()

    pd.DataFrame(shap_validation).to_csv(tables_dir / 'shap_sample_validation.csv', index=False)
    print("✅ SHAP validation saved")

except FileNotFoundError:
    print("⚠️ SHAP values not found - skipping validation")
    shap_validation = None

sys.stdout.flush()

## SAVE ALL RESULTS

In [ ]:
final_results = {
    'architecture_comparison': all_arch_results,
    'shap_validation': shap_validation,
}

with open(tables_dir / 'additional_experiment_results.json', 'w') as f:
    json.dump(final_results, f, indent=2, default=str)

print("\n✅ ALL ADDITIONAL EXPERIMENTS COMPLETE")
sys.stdout.flush()